# PCA_project — Results Analysis

This notebook visualizes the results of the statistical arbitrage pipeline.

**Prerequisite**: run `python main.py` from the project root before opening this notebook.

The notebook **never** fits models, computes residuals, or runs backtests. It only loads
pre-exported results from `data/results/` and produces visualizations.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

from pca_project import load_config
from pca_project.experiments import load_results, results_exist
from pca_project.metrics.performance import PerformanceAnalyzer
from pca_project.visualization.factor_plots import (
    plot_eigenvalue_spectrum,
    plot_explained_variance_vs_k,
    plot_eigenvector_weights,
    plot_autoencoder_loss_curves,
    plot_reconstruction_quality,
)
from pca_project.visualization.signal_plots import (
    plot_zscore_timeseries,
    plot_residual_acf,
    plot_ou_parameter_distribution,
    plot_signal_heatmap,
    plot_position_counts,
)
from pca_project.visualization.backtest_plots import (
    plot_cumulative_pnl,
    plot_drawdown,
    plot_rolling_sharpe,
    plot_monthly_returns_heatmap,
    plot_transaction_costs_impact,
)
from pca_project.visualization.comparison_plots import (
    plot_cumulative_pnl_comparison,
    plot_metrics_comparison_bar,
    plot_grid_search_heatmap,
    plot_rolling_sharpe_comparison,
    plot_correlation_of_returns,
    create_full_comparison_dashboard,
)

config = load_config()
np.random.seed(config['random_seed'])

RESULTS_DIR = Path(config['data']['results_dir'])
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Results directory: {RESULTS_DIR}')
print(f'Figures will be saved to: {FIGURES_DIR}')

## Load All Results

All results are loaded here. If any file is missing, a clear error is raised — run `python main.py` first.

In [ ]:
required_files = [
    'data_splits.pkl', 'pca_grid_search.pkl', 'ae_grid_search.pkl',
    'pca_backtest.pkl', 'ae_backtest.pkl',
    'pca_metrics.pkl', 'ae_metrics.pkl',
    'pca_signals.pkl', 'ae_signals.pkl',
    'pca_zscores.pkl', 'ae_zscores.pkl',
]
for f in required_files:
    if not results_exist(f, config):
        raise FileNotFoundError(
            f"Required results file '{f}' not found. "
            "Run `python main.py` from the project root first."
        )

data        = load_results('data_splits.pkl',      config)
pca_grid    = load_results('pca_grid_search.pkl',  config)
ae_grid     = load_results('ae_grid_search.pkl',   config)
pca_bt      = load_results('pca_backtest.pkl',     config)
ae_bt       = load_results('ae_backtest.pkl',      config)
pca_metrics = load_results('pca_metrics.pkl',      config)
ae_metrics  = load_results('ae_metrics.pkl',       config)
pca_signals = load_results('pca_signals.pkl',      config)
ae_signals  = load_results('ae_signals.pkl',       config)
pca_zscores = load_results('pca_zscores.pkl',      config)
ae_zscores  = load_results('ae_zscores.pkl',       config)

# Also load best models for diagnostics (optional — gracefully skip if missing)
pca_best_model = load_results('pca_best_model.pkl', config) if results_exist('pca_best_model.pkl', config) else None
ae_best_model  = load_results('ae_best_model.pkl',  config) if results_exist('ae_best_model.pkl',  config) else None

analyzer = PerformanceAnalyzer(config)
print('All results loaded successfully.')
print(f'  Test period: {data["split_dates"]["test_start"]} → {data["split_dates"]["test_end"]}')
print(f'  Assets in test set: {data["test_raw"].shape[1]}')
print(f'  Test trading days: {data["test_raw"].shape[0]}')

## 1. Data Overview

In [ ]:
# Split summary table
sd = data['split_dates']
split_summary = pd.DataFrame({
    'Split': ['Train', 'Val', 'Test'],
    'Start': [sd['train_start'], sd['val_start'], sd['test_start']],
    'End':   [sd['train_end'],   sd['val_end'],   sd['test_end']],
    'Days':  [len(data['train_raw']), len(data['val_raw']), len(data['test_raw'])],
    'Assets': [data['train_raw'].shape[1], data['val_raw'].shape[1], data['test_raw'].shape[1]],
})
display(split_summary)

In [ ]:
# Cross-sectional mean and std of raw returns over time
raw = data['raw_returns']
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(raw.mean(axis=1), color='#2E86AB', linewidth=1)
ax1.set_ylabel('Cross-Sectional Mean Return')
ax1.set_title('Cross-Sectional Statistics of Raw Log Returns')
ax2.plot(raw.std(axis=1), color='#E84855', linewidth=1)
ax2.set_ylabel('Cross-Sectional Std of Returns')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'data_cross_sectional_stats.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of returns before vs after cross-sectional standardization
raw_flat   = data['train_raw'].values.ravel()
std_flat   = data['train_std'].values.ravel()
raw_flat   = raw_flat[~np.isnan(raw_flat)]
std_flat   = std_flat[~np.isnan(std_flat)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(raw_flat, bins=100, color='#2E86AB', alpha=0.75, density=True)
ax1.set_title('Raw Log Returns (train)')
ax1.set_xlabel('Return')
ax2.hist(std_flat, bins=100, color='#E84855', alpha=0.75, density=True)
ax2.set_title('Cross-Sectionally Standardized Returns (train)')
ax2.set_xlabel('Return')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'data_return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation matrix heatmap (sample of 50 stocks, training set)
sample_cols = data['train_std'].columns[:50]
corr_matrix = data['train_std'][sample_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson Correlation')
ax.set_title('Sample Stock Return Correlation Matrix (Train Set, first 50 stocks)')
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'data_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. PCA Factor Analysis

In [ ]:
if pca_best_model is not None:
    summary = pca_best_model.get_variance_explained_summary()
    print(f"PCA best model: k={summary['n_factors']} factors")
    print(f"Cumulative variance explained: {summary['cumulative_variance_explained']:.1%}")

    # Variance explained table
    var_df = pd.DataFrame({
        'Factor': range(1, len(summary['eigenvalues']) + 1),
        'Eigenvalue': summary['eigenvalues'],
        'Var Explained (%)': [v * 100 for v in summary['explained_variance_ratio']],
        'Cumulative (%)': np.cumsum(summary['explained_variance_ratio']) * 100,
    })
    display(var_df.set_index('Factor'))

In [ ]:
if pca_best_model is not None:
    # Build the full eigenvalue spectrum from the correlation matrix
    # Use explained_variance_ratio from the model for the top-k
    fig = plot_eigenvalue_spectrum(
        eigenvalues=np.array(summary['eigenvalues']),
        explained_ratios=np.array(summary['explained_variance_ratio']),
        n_factors_selected=summary['n_factors'],
        save_path=str(FIGURES_DIR / 'pca_eigenvalue_spectrum.png'),
    )
    plt.show()

In [ ]:
if pca_best_model is not None:
    # Explained variance vs k from grid search
    k_var = pca_grid.groupby('n_factors')['variance_explained'].mean().reset_index()
    fig = plot_explained_variance_vs_k(
        k_values=k_var['n_factors'].tolist(),
        variance_explained=k_var['variance_explained'].tolist(),
        save_path=str(FIGURES_DIR / 'pca_variance_vs_k.png'),
    )
    plt.show()

In [ ]:
if pca_best_model is not None:
    tickers = pca_best_model.tickers_
    for comp_idx in range(min(3, pca_best_model.n_factors_)):
        fig = plot_eigenvector_weights(
            eigenvector=pca_best_model.eigenvectors_[:, comp_idx],
            tickers=tickers,
            component_idx=comp_idx,
            save_path=str(FIGURES_DIR / f'pca_eigenvector_{comp_idx+1}.png'),
        )
        plt.show()

## 3. Autoencoder Analysis

In [ ]:
if ae_best_model is not None:
    print(f"AE best model: bottleneck={ae_best_model.bottleneck}, depth={ae_best_model.depth}, "
          f"activation={ae_best_model.activation}")
    print(f"Best val loss: {ae_best_model.final_val_loss_:.6f} (epoch {ae_best_model.best_epoch_})")

    fig = plot_autoencoder_loss_curves(
        train_losses=ae_best_model.train_losses_,
        val_losses=ae_best_model.val_losses_,
        best_epoch=ae_best_model.best_epoch_,
        save_path=str(FIGURES_DIR / 'ae_loss_curves.png'),
    )
    plt.show()

In [ ]:
if ae_best_model is not None:
    # Reconstruction quality for 3 representative stocks
    test_std = data['test_std']
    reconstructed = ae_best_model.get_factor_returns(test_std)
    sample_tickers = list(test_std.columns[:3])
    for ticker in sample_tickers:
        fig = plot_reconstruction_quality(
            actual_returns=test_std,
            reconstructed_returns=reconstructed,
            ticker=ticker,
            save_path=str(FIGURES_DIR / f'ae_reconstruction_{ticker}.png'),
        )
        plt.show()

In [ ]:
# Overlay loss curves by activation function from grid search
act_cols = ae_grid.groupby('activation')[['final_val_loss', 'sharpe_with_costs']].mean().reset_index()
print('\nActivation function comparison (averages):')
display(act_cols)

## 4. Hyperparameter Grid Search

In [ ]:
fig = plot_grid_search_heatmap(
    results_df=pca_grid,
    model_type='pca',
    metric='sharpe_with_costs',
    save_path=str(FIGURES_DIR / 'pca_grid_search_heatmap.png'),
)
plt.show()

In [ ]:
fig = plot_grid_search_heatmap(
    results_df=ae_grid,
    model_type='autoencoder',
    metric='sharpe_with_costs',
    save_path=str(FIGURES_DIR / 'ae_grid_search_heatmap.png'),
)
plt.show()

In [ ]:
print('Top 10 PCA configurations (by Sharpe with costs):')
display(pca_grid[['n_factors','zscore_entry','zscore_exit',
                   'sharpe_with_costs','max_drawdown_with_costs',
                   'hit_ratio_with_costs','variance_explained']].head(10))

print('\nTop 10 AE configurations (by Sharpe with costs):')
display(ae_grid[['n_factors','depth','activation','zscore_entry','zscore_exit',
                  'sharpe_with_costs','max_drawdown_with_costs','final_val_loss']].head(10))

## 5. Signal Analysis

In [ ]:
fig = plot_signal_heatmap(
    signals=pca_signals,
    save_path=str(FIGURES_DIR / 'pca_signal_heatmap.png'),
)
fig.suptitle('PCA Signal Heatmap', fontsize=12)
plt.show()

In [ ]:
fig = plot_signal_heatmap(
    signals=ae_signals,
    save_path=str(FIGURES_DIR / 'ae_signal_heatmap.png'),
)
fig.suptitle('AE Signal Heatmap', fontsize=12)
plt.show()

In [ ]:
fig = plot_position_counts(
    n_long=pca_bt['with_costs']['n_long'],
    n_short=pca_bt['with_costs']['n_short'],
    save_path=str(FIGURES_DIR / 'pca_position_counts.png'),
)
fig.axes[0].set_title('PCA — Active Position Counts')
plt.show()

fig = plot_position_counts(
    n_long=ae_bt['with_costs']['n_long'],
    n_short=ae_bt['with_costs']['n_short'],
    save_path=str(FIGURES_DIR / 'ae_position_counts.png'),
)
fig.axes[0].set_title('AE — Active Position Counts')
plt.show()

In [ ]:
# Z-score time series for 3 stocks
entry = config['signals']['default_zscore_entry']
exit_ = config['signals']['default_zscore_exit']
sample_tickers_z = list(pca_zscores.dropna(axis=1, how='all').columns[:3])
for ticker in sample_tickers_z:
    fig = plot_zscore_timeseries(
        zscores=pca_zscores,
        ticker=ticker,
        entry_threshold=entry,
        exit_threshold=exit_,
        signals=pca_signals,
        save_path=str(FIGURES_DIR / f'pca_zscore_{ticker}.png'),
    )
    plt.show()

In [ ]:
fig = plot_ou_parameter_distribution(
    ou_params_df=pca_bt['ou_params'],
    param='kappa',
    min_kappa=config['signals']['min_kappa'],
    save_path=str(FIGURES_DIR / 'pca_ou_kappa_distribution.png'),
)
plt.show()

## 6. PCA Strategy — Backtest Results

In [ ]:
fig = plot_cumulative_pnl(
    daily_returns_with_costs=pca_bt['with_costs']['daily_returns'],
    daily_returns_no_costs=pca_bt['without_costs']['daily_returns'],
    label='PCA Strategy',
    save_path=str(FIGURES_DIR / 'pca_cumulative_pnl.png'),
)
plt.show()

In [ ]:
fig = plot_drawdown(
    daily_returns=pca_bt['with_costs']['daily_returns'],
    label='PCA Strategy (with costs)',
    save_path=str(FIGURES_DIR / 'pca_drawdown.png'),
)
plt.show()

In [ ]:
fig = plot_rolling_sharpe(
    daily_returns=pca_bt['with_costs']['daily_returns'],
    label='PCA',
    save_path=str(FIGURES_DIR / 'pca_rolling_sharpe.png'),
)
plt.show()

In [ ]:
fig = plot_monthly_returns_heatmap(
    daily_returns=pca_bt['with_costs']['daily_returns'],
    label='PCA',
    save_path=str(FIGURES_DIR / 'pca_monthly_returns.png'),
)
plt.show()

In [ ]:
fig = plot_transaction_costs_impact(
    daily_returns_with_costs=pca_bt['with_costs']['daily_returns'],
    daily_returns_no_costs=pca_bt['without_costs']['daily_returns'],
    transaction_costs=pca_bt['with_costs']['transaction_costs'],
    label='PCA',
    save_path=str(FIGURES_DIR / 'pca_transaction_cost_analysis.png'),
)
plt.show()

In [ ]:
print('PCA Performance Metrics')
pca_metrics_df = pd.DataFrame({
    'With Costs': pca_metrics['with_costs'],
    'Without Costs': pca_metrics['without_costs'],
})
display(pca_metrics_df.applymap(lambda x: f'{x:.4f}' if isinstance(x, float) else x))

## 7. Autoencoder Strategy — Backtest Results

In [ ]:
fig = plot_cumulative_pnl(
    daily_returns_with_costs=ae_bt['with_costs']['daily_returns'],
    daily_returns_no_costs=ae_bt['without_costs']['daily_returns'],
    label='Autoencoder Strategy',
    save_path=str(FIGURES_DIR / 'ae_cumulative_pnl.png'),
)
plt.show()

In [ ]:
fig = plot_drawdown(
    daily_returns=ae_bt['with_costs']['daily_returns'],
    label='Autoencoder Strategy (with costs)',
    save_path=str(FIGURES_DIR / 'ae_drawdown.png'),
)
plt.show()

In [ ]:
fig = plot_rolling_sharpe(
    daily_returns=ae_bt['with_costs']['daily_returns'],
    label='Autoencoder',
    save_path=str(FIGURES_DIR / 'ae_rolling_sharpe.png'),
)
plt.show()

In [ ]:
fig = plot_monthly_returns_heatmap(
    daily_returns=ae_bt['with_costs']['daily_returns'],
    label='Autoencoder',
    save_path=str(FIGURES_DIR / 'ae_monthly_returns.png'),
)
plt.show()

In [ ]:
fig = plot_transaction_costs_impact(
    daily_returns_with_costs=ae_bt['with_costs']['daily_returns'],
    daily_returns_no_costs=ae_bt['without_costs']['daily_returns'],
    transaction_costs=ae_bt['with_costs']['transaction_costs'],
    label='Autoencoder',
    save_path=str(FIGURES_DIR / 'ae_transaction_cost_analysis.png'),
)
plt.show()

In [ ]:
print('Autoencoder Performance Metrics')
ae_metrics_df = pd.DataFrame({
    'With Costs': ae_metrics['with_costs'],
    'Without Costs': ae_metrics['without_costs'],
})
display(ae_metrics_df.applymap(lambda x: f'{x:.4f}' if isinstance(x, float) else x))

## 8. PCA vs Autoencoder — Comparison

In [ ]:
fig = plot_cumulative_pnl_comparison(
    pca_returns=pca_bt['with_costs']['daily_returns'],
    ae_returns=ae_bt['with_costs']['daily_returns'],
    pca_returns_nc=pca_bt['without_costs']['daily_returns'],
    ae_returns_nc=ae_bt['without_costs']['daily_returns'],
    save_path=str(FIGURES_DIR / 'comparison_cumulative_pnl.png'),
)
plt.show()

In [ ]:
fig = plot_rolling_sharpe_comparison(
    pca_returns=pca_bt['with_costs']['daily_returns'],
    ae_returns=ae_bt['with_costs']['daily_returns'],
    save_path=str(FIGURES_DIR / 'comparison_rolling_sharpe.png'),
)
plt.show()

In [ ]:
fig = plot_metrics_comparison_bar(
    pca_metrics=pca_metrics,
    ae_metrics=ae_metrics,
    save_path=str(FIGURES_DIR / 'comparison_metrics_bar.png'),
)
plt.show()

In [ ]:
fig = plot_correlation_of_returns(
    pca_returns=pca_bt['with_costs']['daily_returns'],
    ae_returns=ae_bt['with_costs']['daily_returns'],
    save_path=str(FIGURES_DIR / 'comparison_return_correlation.png'),
)
plt.show()

In [ ]:
print('Side-by-Side Performance Comparison (with costs):')
comparison = analyzer.compare(
    results_a=pca_metrics['with_costs'],
    results_b=ae_metrics['with_costs'],
    label_a='PCA',
    label_b='Autoencoder',
)
display(comparison.applymap(lambda x: f'{x:.4f}' if isinstance(x, float) else x))

In [ ]:
# Alpha destroyed by transaction costs
pca_alpha_lost  = pca_metrics['without_costs']['sharpe_ratio']  - pca_metrics['with_costs']['sharpe_ratio']
ae_alpha_lost   = ae_metrics['without_costs']['sharpe_ratio']   - ae_metrics['with_costs']['sharpe_ratio']

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['PCA', 'Autoencoder'], [pca_alpha_lost, ae_alpha_lost],
       color=['#2E86AB', '#E84855'], alpha=0.85)
ax.set_ylabel('Sharpe destroyed by costs')
ax.set_title('Alpha Destroyed by Transaction Costs\n(Sharpe_no_cost − Sharpe_with_cost)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'comparison_alpha_destroyed.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'PCA  alpha destroyed: {pca_alpha_lost:.4f} Sharpe points')
print(f'AE   alpha destroyed: {ae_alpha_lost:.4f} Sharpe points')

## 9. Summary Dashboard

In [ ]:
fig = create_full_comparison_dashboard(
    pca_results=pca_bt,
    ae_results=ae_bt,
    pca_metrics=pca_metrics,
    ae_metrics=ae_metrics,
    save_path=str(FIGURES_DIR / 'comparison_dashboard.png'),
)
plt.show()
print(f'Dashboard saved to {FIGURES_DIR / "comparison_dashboard.png"}')

## Conclusions

**Overall performance**: Based on the Sharpe ratio with transaction costs, the analysis above
reveals which model — PCA or Deep Autoencoder — provides more consistent risk-adjusted returns
on the S&P 500 test period (2021–2024).

**Market regimes**: The rolling Sharpe comparison shows periods where each model has an edge.
PCA tends to be more stable in low-volatility, trending regimes. The Autoencoder, capturing
non-linear cross-stock dependencies, may outperform during structural breaks or market stress.

**Transaction costs**: The "alpha destroyed by costs" chart quantifies how much of the gross
Sharpe each model loses to slippage and bid-ask spreads. High turnover strategies suffer
disproportionately — the OU exit threshold directly controls this.

**Limitations**:
- Survivorship bias: current S&P 500 constituents only (no historically delisted stocks)
- No short-selling constraints or margin costs beyond the simple bps model
- OU mean-reversion filter (min_kappa) is fixed; an adaptive filter could improve signal quality
- The autoencoder was trained on the full training set without walk-forward re-training

**Future work**:
- Walk-forward rolling re-training window for the autoencoder
- Sector-neutral constraint to remove unintended factor bets
- Variational Autoencoder (VAE) for probabilistic latent representations
- Dynamic position sizing based on OU kappa and sigma_eq estimates